# 🔬 Phase 2 (M1): Offline Calibration & Audit — 32B Local LLM Judge + Python/MiniLM Constraints

Kiểm định hệ thống **Expansion Reward** (trên **1× A100 80GB, 0 phí API**) **TRƯỚC KHI TRAIN GRPO**:
- **Phần dùng LLM 32B (`unsloth/Qwen2.5-32B-Instruct-bnb-4bit`, ~19.5 GB VRAM)**:
  - **Level 1 (Local Expansion)**: `event_realization`, `causal_development`, `meaningful_elaboration`
  - **Level 2 (Global Chapter)**: `macro_adherence`, `chapter_coherence`, `character_consistency`, `narrative_progression` + `contradictions`
- **Phần dùng Python + MiniLM (0 token LLM)**:
  - **Level 3 (Constraints)**: `P_length`, `P_rep` (MiniLM cosine + dedup + anti-context), `P_contra` (công thức trọng số lỗi), `P_format` (`<think>`), và **Reward Aggregator** (`EXP-00 .. EXP-05`).
- Kiểm tra 3 điều kiện **Go/No-Go**: (1) Không bão hòa điểm, (2) Tương quan Spearman `Local` vs `Global` `< 0.88` (H2), (3) `Clean` thắng `Padded-Filler`, `Off-Plan`, và `Contradictory`.

In [ ]:
# 1. Mount Google Drive & Sync MA-GRPO Repo
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/MA-GRPO')
REPO_URL = 'github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git'
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN') or os.environ.get('GH_TOKEN', '')
except Exception:
    GH_TOKEN = os.environ.get('GH_TOKEN', '')

clone_url = f'https://{GH_TOKEN}@{REPO_URL}' if GH_TOKEN else f'https://{REPO_URL}'
if not REPO_DIR.exists():
    !git clone {clone_url} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 2. Install Dependencies (bitsandbytes + transformers + sentence-transformers)
!pip install -q transformers accelerate bitsandbytes sentence-transformers pyyaml matplotlib pandas

In [ ]:
# 3. Configure Input Stories & Phase 2 Reward Config
STORIES_JSONL = '/content/drive/MyDrive/MA-GRPO-ContextGate-Ablation/stories_7b_outputs.jsonl'
REWARD_CONFIG = 'rewards/configs/exp02_hierarchical.yaml'
OUTPUT_DIR = '/content/drive/MyDrive/MA-GRPO-Phase2-Audit/exp02_32b_audit'
MAX_CHAPTERS = 16
PERTURBATION_CHAPTERS = 4
AUTO_DISCONNECT = False

In [ ]:
# 4. Run M1 Offline Audit (Separate Local & Global 32B passes to verify H2 without halo effect)
import subprocess
import sys

cmd = [
    sys.executable, '-u', 'scripts/audit_rewards_offline.py',
    '--stories', STORIES_JSONL,
    '--reward-config', REWARD_CONFIG,
    '--output-dir', OUTPUT_DIR,
    '--max-chapters', str(MAX_CHAPTERS),
    '--perturbation-chapters', str(PERTURBATION_CHAPTERS),
]
print('Running:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'audit_rewards_offline.py exited with code {ret}')

In [ ]:
# 5. Visualize Audit Report, Score Distributions & 4-Way Perturbation Discrimination
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

out = Path(OUTPUT_DIR)
display(Markdown((out / 'offline_audit_report.md').read_text(encoding='utf-8')))

summary = json.loads((out / 'offline_audit_summary.json').read_text(encoding='utf-8'))
pert_df = pd.DataFrame(summary['perturbation_discrimination']['details'])
display(pert_df)

means = pert_df[['r_clean', 'r_padded_filler', 'r_off_plan', 'r_contradictory']].mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(['Clean', 'Padded Filler', 'Off-Plan', 'Contradictory'], means.values, color=['#2f855a', '#dd6b20', '#c53030', '#805ad5'])
ax.set_ylabel('Mean R_total')
ax.set_title('Phase 2 Reward Discrimination (32B Judge + Python Constraints)')
plt.tight_layout()
plt.savefig(out / 'perturbation_discrimination.png', dpi=150)
plt.show()

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()